# Problema Direto Transitório 

Este problema tem como objetivo implementar uma PINN para um problema direto transitório.

**Autor:** Edélio Gabriel Magalhães de Jesus

## Definição do problema

O primeiro problema que discutimos tratava-se de um problema apenas de valor de contorno — logo, não considerávamos nenhuma evolução temporal. Por isso o chamamos de estacionário.

Agora, vamos adicionar um novo aspecto: as **condições iniciais**. Elas fundamentam o caráter temporal do problema — a solução não é mais um estado de equilíbrio, mas uma função que evolui no tempo a partir de um estado inicial conhecido.

Para isso, trabalharemos com a **Equação de Burgers**.

---

### `Requisitos teóricos`

####
Equação de Burgers

A equação de Burgers é uma equação diferencial parcial parabólica do tipo convecção-difusão (Equação 1), que descreve o transporte de uma grandeza física por dois mecanismos simultâneos: **convecção** — o transporte pelo próprio campo — e **difusão** — o espalhamento devido à viscosidade [[ref]](#eq-convec-difu). Ela aparece em diversas áreas da matemática aplicada e da física, como mecânica dos fluidos, acústica não linear, dinâmica de gases e fluxo de tráfego.

$$
u_t + c\cdot u_x = \mu\cdot u_{xx} \tag{1}
$$

O aspecto mais discutido da equação de Burgers é a competição entre esses dois mecanismos: a convecção tende a acentuar gradientes e formar **choques** — descontinuidades abruptas na solução — enquanto a difusão os suaviza. Para viscosidades pequenas, os choques dominam e a solução desenvolve frentes muito íngremes, o que torna o problema numericamente desafiador e um benchmark clássico para métodos numéricos e, mais recentemente, para PINNs [[ref]](#original-paper).

<div style="text-align: center;">
  <img 
    src="https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/Airplane_vortex.jpg/1280px-Airplane_vortex.jpg" 
    alt="Ilustração da mecânica de fluidos"
    style="max-width: 400px; width: 100%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://pt.wikipedia.org/wiki/Mec%C3%A2nica_dos_fluidos' target='_blank'>
    Wikipedia — Mecânica dos fluidos
  </a>
</div>

Ela é expressa por:

$$
u_t + u \cdot u_x = \nu \cdot u_{xx} \tag{2}
$$

onde $u(x, t)$ é o campo de velocidade, $\nu > 0$ é a viscosidade cinemática, $u \cdot u_x$ é o termo de **convecção não linear** e $\nu \cdot u_{xx}$ é o termo de **difusão**. Note que, diferentemente da equação de advecção-difusão linear, o coeficiente de convecção aqui é a própria solução $u$ — o que torna o problema não linear.

---

## Aplicando a PINN

O código completo está localizado na pasta `scripts`, especificamente no arquivo `ex03_pinn_direct_transient.py`. Para facilitar a discussão, colocarei apenas trechos necessários para uma compreensão mais aprofundada.

---

A célula seguinte serve para:

- Recarregar automaticamente qualquer arquivo que for editado nos scripts
- Encontrar a pasta dos *scripts*, permitindo importar as funções criadas

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath("../scripts"))

import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

### **Importações necessárias**

In [2]:
import torch.nn as nn
import torch.optim as optim
import torch
import plotly.graph_objects as go
from geral_functions import PINN, sample_collocation_rectangular, sample_boundary_rectangular_transient
from plot_utils import plot_loss, plot_heatmaps, plot_points_transient, plot_profiles
from ex03_pinn_direct_transient import train_burgers, evaluate_burgers, numerical_solution_burgers


### **Parâmetros do problema**

Os valores dos parâmetros que envolvem a arquitetura da rede a amostragem foram inspirados na discussão presente artigo original de "Raissi et. al. ("**Data-driven solutions of nonlinear partial differential equations**"[[ref]](#original-paper)), em que uma PINN foi implementada para esse meesmo problema.

In [3]:
# Definição do local onde o código serpa executado. Por padrão, gpu
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {DEVICE}')

# Arquitetura da rede
N_INPUTS = 2
N_OUTPUTS = 1
N_HIDDEN = 16
N_LAYERS = 9
ACTIVATION = nn.Tanh

# Parâmetros do problema
X_LB, X_UB = -1.0, 1.0
T_LB, T_UB = 0.0, 1.0
NU = 0.01 / torch.pi

IC_FN = lambda x: -torch.sin(torch.pi * x)

BC_FNS = {
    'left':  lambda t: torch.zeros_like(t),
    'right': lambda t: torch.zeros_like(t),
}

# Parâmetros de amostragem
N_COLLOC = 5000
N_BC = 150
N_IC = 150

# Parâmetros do treinamento
W_IC = 1.0
W_BC = 1.0
W_PDE = 1.0
N_EPOCHS = 10000
LR = 1e-4


Usando: cpu


### **Instanciando o modelo**

In [4]:
model = PINN(N_INPUTS, N_OUTPUTS, N_HIDDEN, N_LAYERS, ACTIVATION)

### **Amostragem dos pontos**

In [6]:
X_COLLOC = sample_collocation_rectangular(N_COLLOC, [X_LB, T_LB], [X_UB, T_UB], DEVICE)

X_IC, U_IC, X_BC, U_BC = sample_boundary_rectangular_transient(
    N_IC, N_BC,
    X_LB, X_UB,
    T_LB, T_UB,
    IC_FN, BC_FNS, DEVICE
)

---

> Vale destacar que estamos usando uma amostragem aleatória, enquanto que os autores optaram por uma amostragem pelo método quase-aleatório *Latin Hypercube Sampling (LHS)* - que é conhecido conseguir alcançar uma cobertura mais uniforme de todo o domínio. 

> Agora, você pode se perguntar: quais são os impactos da amostragem no aprendizado do nosso modelo?

> Se ficou curioso, confira o *notebook* `07_sampling.ipynb`!

---

### **Instanciando o otimizador**

In [7]:
OPTIMIZER = torch.optim.Adam(model.parameters(), lr=LR)

### **Treinamento**

In [8]:
history = train_burgers(model, OPTIMIZER, X_COLLOC, X_IC, U_IC, X_BC, U_BC, NU, N_EPOCHS, W_IC, W_BC, W_PDE)

Epoch 00000 | Loss: 5.11e-01 | Loss IC: 5.06e-01 | Loss BC: 4.48e-03 | Loss PDE: 3.31e-06
Epoch 00100 | Loss: 4.69e-01 | Loss IC: 4.67e-01 | Loss BC: 2.52e-03 | Loss PDE: 1.99e-04
Epoch 00200 | Loss: 4.15e-01 | Loss IC: 3.92e-01 | Loss BC: 1.55e-02 | Loss PDE: 7.83e-03
Epoch 00300 | Loss: 3.82e-01 | Loss IC: 3.11e-01 | Loss BC: 5.87e-02 | Loss PDE: 1.25e-02
Epoch 00400 | Loss: 3.65e-01 | Loss IC: 2.82e-01 | Loss BC: 7.60e-02 | Loss PDE: 7.04e-03
Epoch 00500 | Loss: 3.42e-01 | Loss IC: 2.52e-01 | Loss BC: 8.42e-02 | Loss PDE: 6.51e-03
Epoch 00600 | Loss: 3.11e-01 | Loss IC: 2.19e-01 | Loss BC: 8.56e-02 | Loss PDE: 6.40e-03
Epoch 00700 | Loss: 2.92e-01 | Loss IC: 2.02e-01 | Loss BC: 8.20e-02 | Loss PDE: 7.94e-03
Epoch 00800 | Loss: 2.82e-01 | Loss IC: 1.97e-01 | Loss BC: 7.40e-02 | Loss PDE: 1.15e-02
Epoch 00900 | Loss: 2.68e-01 | Loss IC: 1.91e-01 | Loss BC: 6.10e-02 | Loss PDE: 1.66e-02
Epoch 01000 | Loss: 2.30e-01 | Loss IC: 1.70e-01 | Loss BC: 3.55e-02 | Loss PDE: 2.45e-02
Epoch 0110

### **Visualizando os resultados do treinamento**

In [18]:
plot_points_transient(X_COLLOC, X_BC, X_IC)

In [10]:
plot_loss(history)

### **Validando o modelo**

Vamos primeiro gerar a solução numérica para o nosso problema

In [11]:
x_ref, t_ref, U_ref = numerical_solution_burgers()

In [12]:
print("len(t_ref):", len(t_ref))
print("U_ref.shape:", U_ref.shape)

len(t_ref): 100
U_ref.shape: (256, 100)


In [13]:
results = evaluate_burgers(
    model=model,
    x_ref=x_ref,
    t_ref=t_ref,
    U_ref=U_ref,
    device=DEVICE
)

In [14]:
plot_heatmaps(
    results['U_pred'], results['U_ref'],
    results['t'], results['x'],
    title='Equação de Burgers',
    xlabel='t', ylabel='x'
)

In [15]:
# snapshots temporais
plot_profiles(
    results['U_pred_snaps'], results['U_ref_snaps'],
    results['x'],
    slices=results['snap_times'],
    title='Snapshots temporais',
    xlabel='x', ylabel='u(x, t)',
    slice_label='t'
)

<a id="original-paper"> </a> [RAISSI, Maziar; PERDIKARIS, Paris; KARNIADAKIS, George Em. Physics informed deep learning (part i): Data-driven solutions of nonlinear partial differential equations. arXiv preprint arXiv:1711.10561, 2017.](https://arxiv.org/abs/1711.10561)

<a id='eq-comvec-difu'></a> [Wikipedia - Equação de convecção-difusão](https://pt.wikipedia.org/wiki/Equa%C3%A7%C3%A3o_de_convec%C3%A7%C3%A3o-difus%C3%A3o)